# 05 — Molecular Docking (Schrödinger Glide)

Dock the top virtual screening hits into the EGFR binding site using Glide:
1. Ligand preparation (RDKit 3D conformer generation)
2. Glide docking (SP precision)
3. Result analysis and scoring
4. Selection of top 2 compounds for MD simulation

**Prerequisites:**
- Schrödinger Suite installed (`$SCHRODINGER` set)
- EGFR receptor grid prepared via Protein Prep Wizard / Glide Grid Generation
- Virtual screening hits from notebook 04

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from rdkit import Chem
from rdkit.Chem import Draw

from src.components.docking import GlideDocking

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)

docker = GlideDocking(config)

## 1. Load Virtual Screening Hits

In [ ]:
hits = pd.read_csv("../results/virtual_screening_hits.csv")
print(f"VS hits: {len(hits)}")
smiles_list = hits["canonical_smiles"].tolist()[:config["docking"]["top_n_for_docking"]]
print(f"Compounds for docking: {len(smiles_list)}")

## 2. Prepare Ligands (3D Conformers)

In [ ]:
sdf_path = docker.prepare_ligands(smiles_list)
print(f"Ligand SDF: {sdf_path}")

## 3. Run Glide Docking

Note: This requires Schrödinger Suite. If not available, placeholder results will be generated.

In [ ]:
input_file = docker.write_glide_input(ligand_file=sdf_path)
output_path = docker.run_glide(input_file)

## 4. Parse & Analyze Results

In [ ]:
results = docker.parse_glide_results()
if not results.empty:
    ranked = docker.rank_by_score(results)
    print(ranked.head(10))

    # GlideScore distribution
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(ranked["GlideScore"], bins=30, kde=True, color="steelblue", ax=ax)
    ax.set_xlabel("GlideScore (kcal/mol)")
    ax.set_ylabel("Count")
    ax.set_title("GlideScore Distribution")
    plt.tight_layout()
    plt.savefig("../results/plots/glide_score_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No docking results available. Ensure Schrödinger Glide is installed.")

## 5. Select Top 2 for MD Simulation

In [ ]:
if not results.empty:
    top_md = docker.select_top_for_md(results)
    print("Top compounds selected for MD simulation:")
    print(top_md)
    docker.export_top_poses(results)

    # Visualize top 2
    if "SMILES" in top_md.columns:
        mols = [Chem.MolFromSmiles(s) for s in top_md["SMILES"]]
        legends = [f"GlideScore={row['GlideScore']:.2f}" for _, row in top_md.iterrows()]
        img = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(400, 400), legends=legends)
        display(img)
else:
    print("No docking results to select from.")